In [8]:
!pip install groq
!pip install dotenv


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\beimn\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\beimn\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
#Set Groq API token
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv()


GROQ_API_KEY loaded


## Define Tools

In [16]:
from geopy.geocoders import Nominatim
import requests

def get_weather(city: str) -> int:
    """ Provide weather information with city as input and temperature as output in degree celsius(integer) """

    #api call

    try:
        # convert city name to latitude and longitude
        geo_locator = Nominatim(user_agent="weather_agent_colab")
        location = geo_locator.geocode(city)
        if not location:
            raise ValueError("City not found")

        lat, lon = location.latitude, location.longitude

        # fetch weather data from open-meteo free API
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # extract temperature
        temperature = int(round(data["current_weather"]["temperature"]))
        return temperature
        
    except Exception as e:
        print(f"Error fetching weather: {e}")
        return 0

In [17]:
# Suggest acctivity 
def suggest_activity(temperature: int) -> str:
    "Suggests activity to do as per temperature as input"
    if temperature < 21:
        return "Eat Bokolo"
    elif temperature > 20 and temperature < 25:
        return "Dance with your girl"
    else:
        return "Jump on bed"

In [19]:
# Create all TOOLS
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "weather",
            "description": (
                "Allows to get weather of a city in degree celsius"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city",
                    }
                },
                "required": ["city"]
            },
        },
    },
    
    {
        "type": "function",
        "function": {
            "name": "suggest_activity",
            "description": (
                "Suggests activity to do as per temperature as input"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "temperature": {
                        "type": "integer",
                        "description": "Temperature of the city",
                    }
                },
                "required": ["temperature"]
            },
        },
    }
]

In [13]:
# Tool functions - maps strings to actual functions
TOOL_FUNCTION = {
    "weather": get_weather,
    "suggest_activity": suggest_activity
}

In [ ]:
import json

QUESTION = "suggest me an activity based on the weather in Addis Ababa"

# Create messages
MESSAGES = [
    {
        "role": "user", "content": QUESTION
    }
]

MAX_STEPS = 5

for step in range(MAX_STEPS):
    
    # Call LLM with TOOL
    from groq import Groq
    client = Groq()
    MODEL = "openai/gpt-oss-120b"

    res = client.chat.completions.create(
        model = MODEL,
        max_tokens = 1000,
        tools = TOOLS,
        messages = MESSAGES,
        parallel_tool_calls = False
    )
    
    # Append Assistant Message
    MESSAGES.append(res.choices[0].message)
    
    if not res.choices[0].message.tool_calls:
        print(f"final Output: {res.choices[0].message.content}")
        break
    
    # Execute the tool to send back observation to model
    for call in res.choices[0].message.tool_calls:
        tool_id =call.id
        func_name =call.function.name
        _args = json.loads(call.function.arguments)

        tool_output = TOOL_FUNCTION[func_name](**_args)
        MESSAGES.append(
            {
                "role": "tool", "content": str(tool_output), "tool_call_id": tool_id, "name": func_name
            }
        )
        print(f"Tool call -> ",  tool_id, func_name, _args, tool_output)

Tool call ->  fc_b5d2755f-96ba-425e-9c81-30a4618d211d weather {'city': 'Addis Ababa'} 16
Tool call ->  fc_bed5d4d7-bfc6-4c78-9777-f255b27ff17b suggest_activity {'temperature': 16} Eat Bokolo
final Output: Based on Addis Ababa’s current temperature of about 16 °C (61 °F), a comfortable, mildly cool day, it’s a great time for a relaxed outdoor activity that isn’t too strenuous. Here are a few ideas:

1. **Stroll through the Entoto Hills** – Enjoy gentle hiking or a leisurely walk amidst the pine forests and historic churches. The cool air and scenic views make it perfect for a relaxed outing.  
2. **Visit the National Museum of Ethiopia** – If you’d rather stay indoors, explore Ethiopia’s rich history, including the famous fossil “Lucy.” It’s a cultural experience that pairs well with a mild temperature.  
3. **Coffee tasting at a local café** – Take advantage of the weather to sit outdoors at a rooftop café and sample Ethiopia’s world‑renowned coffee while people‑watching.  
4. **Picnic